# Article Figures Generator
**Purpose:** Read existing CSV results and generate publication-ready PDF figures.

**Requirements:**
- Color-blind friendly palette (Wong's)
- Bold axis labels
- PDF output format
- Line charts (no bar charts)

**Usage:** Run all cells. PDFs are saved to the same folder as this notebook.


In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")  # non-interactive backend (Colab-safe)
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

# ── Wong's color-blind safe palette ──────────────────────────────────
WONG = {
    "blue":    "#0072B2",
    "orange":  "#E69F00",
    "green":   "#009E73",
    "yellow":  "#F0E442",
    "skyblue": "#56B4E9",
    "red":     "#D55E00",
    "pink":    "#CC79A7",
    "black":   "#000000",
}
WONG_LIST = list(WONG.values())

# ── Global matplotlib style ──────────────────────────────────────────
plt.rcParams.update({
    "font.family": "serif",
    "font.size": 12,
    "axes.labelweight": "bold",
    "axes.labelsize": 13,
    "axes.titleweight": "bold",
    "axes.titlesize": 14,
    "xtick.labelsize": 11,
    "ytick.labelsize": 11,
    "legend.fontsize": 10,
    "figure.dpi": 300,
    "savefig.bbox": "tight",
    "savefig.pad_inches": 0.1,
})

# ── Paths (edit these if running in Colab) ───────────────────────────
# When running locally from article_figures/:
BASE = os.path.dirname(os.path.abspath("__file__"))  # notebook folder
PROJECT = os.path.join(BASE, "..")  # homogenity_project root

SO_ABLATION_CSV   = os.path.join(PROJECT, "ablation_results", "so_ablation_summary.csv")
ACS_ABLATION_CSV  = os.path.join(PROJECT, "ablation_results", "acs_ablation_summary.csv")
BENCH_P2_CSV      = os.path.join(PROJECT, "problem_2_3_algorithms", "benchmark_two_phase_FIXED",
                                 "problem2_largest_delta", "find_delta_benchmark_results.csv")
BENCH_P3_CSV      = os.path.join(PROJECT, "problem_2_3_algorithms", "benchmark_two_phase_FIXED",
                                 "problem3_smallest_epsilon", "find_epsilon_benchmark_results.csv")

OUTPUT_DIR = BASE  # PDFs saved next to this notebook

print("✅ Setup complete")
print(f"   SO ablation CSV:  {os.path.exists(SO_ABLATION_CSV)}")
print(f"   ACS ablation CSV: {os.path.exists(ACS_ABLATION_CSV)}")
print(f"   Benchmark P2 CSV: {os.path.exists(BENCH_P2_CSV)}")
print(f"   Benchmark P3 CSV: {os.path.exists(BENCH_P3_CSV)}")


In [ ]:
# ── Helper: save figure as PDF ────────────────────────────────────────
def save_pdf(fig, name):
    path = os.path.join(OUTPUT_DIR, name)
    fig.savefig(path, format="pdf")
    plt.close(fig)
    print(f"  ✅ Saved: {name}")


# ── Helper: make a short readable rule label from dict-style strings ─
def _short_rule_label(condition_str, treatment_str):
    """Turn \"{'Gender': 'Male'}\" -> \"Gender=Male\" etc."""
    import ast
    try:
        cond = ast.literal_eval(condition_str)
        treat = ast.literal_eval(treatment_str)
    except Exception:
        return f"{condition_str} → {treatment_str}"
    cond_parts = [f"{k}={v}" for k, v in cond.items()]
    treat_parts = [f"{k}={v}" for k, v in treat.items()]
    # Truncate long values
    def _trunc(s, max_len=25):
        return s if len(s) <= max_len else s[:max_len-1] + "…"
    cond_s = ", ".join(_trunc(p) for p in cond_parts)
    treat_s = ", ".join(_trunc(p) for p in treat_parts)
    return f"{cond_s} → {treat_s}"


# ── Helper: generate all 6 ablation figures for one dataset ──────────
def generate_ablation_figures(csv_path, ds_prefix, ds_label):
    """Generate 6 ablation PDFs for a given dataset."""
    df = pd.read_csv(csv_path)

    eps_data   = df[df["experiment"] == "varying_epsilon"]
    delta_data = df[df["experiment"] == "varying_delta"]

    # ── Extract series ────────────────────────────────────────────────
    eps_agreement  = eps_data[eps_data["algorithm"] == "RW_Agreement"].sort_values("epsilon")
    delta_agreement = delta_data[delta_data["algorithm"] == "RW_Agreement"].sort_values("delta_pct")

    eps_fp  = eps_data[eps_data["algorithm"] == "FPGrowth"].sort_values("epsilon")
    eps_rw  = eps_data[eps_data["algorithm"] == "RW_Direct"].sort_values("epsilon")
    delta_fp = delta_data[delta_data["algorithm"] == "FPGrowth"].sort_values("delta_pct")
    delta_rw = delta_data[delta_data["algorithm"] == "RW_Direct"].sort_values("delta_pct")

    # ── 1. RW Correctness vs Epsilon ──────────────────────────────────
    fig, ax = plt.subplots(figsize=(7, 4.5))
    ax.plot(eps_agreement["epsilon"], eps_agreement["agreement_rate"],
            marker="o", linewidth=2.5, color=WONG["blue"], label="RW Correctness")
    ax.fill_between(eps_agreement["epsilon"], eps_agreement["agreement_rate"],
                    alpha=0.15, color=WONG["blue"])
    ax.set_xlabel("Epsilon")
    ax.set_ylabel("Agreement with FPGrowth (%)")
    ax.set_ylim(0, 105)
    ax.set_title(f"RW Agreement Rate with FPGrowth vs Epsilon ({ds_label})")
    ax.legend()
    ax.grid(True, alpha=0.3)
    save_pdf(fig, f"ablation_{ds_prefix}_correctness_varying_epsilon.pdf")

    # ── 2. RW Correctness vs Delta ────────────────────────────────────
    fig, ax = plt.subplots(figsize=(7, 4.5))
    ax.plot(delta_agreement["delta_pct"], delta_agreement["agreement_rate"],
            marker="o", linewidth=2.5, color=WONG["green"], label="RW Correctness")
    ax.fill_between(delta_agreement["delta_pct"], delta_agreement["agreement_rate"],
                    alpha=0.15, color=WONG["green"])
    ax.set_xlabel("Delta (% of Dataset)")
    ax.set_ylabel("Agreement with FPGrowth (%)")
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:.0f}%"))
    ax.set_ylim(0, 105)
    ax.set_title(f"RW Agreement Rate with FPGrowth vs Delta ({ds_label})")
    ax.legend()
    ax.grid(True, alpha=0.3)
    save_pdf(fig, f"ablation_{ds_prefix}_correctness_varying_delta.pdf")

    # ── 3. Runtime vs Epsilon (LINE chart) ────────────────────────────
    fig, ax = plt.subplots(figsize=(7, 4.5))
    ax.plot(eps_fp["epsilon"], eps_fp["runtime_seconds_mean"],
            marker="s", linewidth=2.5, color=WONG["blue"], label="FPGrowth")
    ax.plot(eps_rw["epsilon"], eps_rw["runtime_seconds_mean"],
            marker="o", linewidth=2.5, color=WONG["orange"], label="RW_Direct")
    ax.set_xlabel("Epsilon")
    ax.set_ylabel("Average Runtime (seconds)")
    ax.set_title(f"Runtime Comparison FPGrowth vs RW — Varying Epsilon ({ds_label})")
    ax.legend()
    ax.grid(True, alpha=0.3)
    save_pdf(fig, f"ablation_{ds_prefix}_runtime_varying_epsilon.pdf")

    # ── 4. Runtime vs Delta (LINE chart) ──────────────────────────────
    fig, ax = plt.subplots(figsize=(7, 4.5))
    ax.plot(delta_fp["delta_pct"], delta_fp["runtime_seconds_mean"],
            marker="s", linewidth=2.5, color=WONG["blue"], label="FPGrowth")
    ax.plot(delta_rw["delta_pct"], delta_rw["runtime_seconds_mean"],
            marker="o", linewidth=2.5, color=WONG["orange"], label="RW_Direct")
    ax.set_xlabel("Delta (% of Dataset)")
    ax.set_ylabel("Average Runtime (seconds)")
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:.0f}%"))
    ax.set_title(f"Runtime Comparison FPGrowth vs RW — Varying Delta ({ds_label})")
    ax.legend()
    ax.grid(True, alpha=0.3)
    save_pdf(fig, f"ablation_{ds_prefix}_runtime_varying_delta.pdf")

    # ── 5. Param Impact — Epsilon ─────────────────────────────────────
    fig, ax = plt.subplots(figsize=(7, 4.5))
    if len(eps_fp) > 0:
        ax.plot(eps_fp["epsilon"], eps_fp["homogeneity_rate"],
                marker="s", linewidth=2.5, color=WONG["blue"], label="FPGrowth")
    if len(eps_rw) > 0:
        ax.plot(eps_rw["epsilon"], eps_rw["homogeneity_rate"],
                marker="o", linewidth=2.5, color=WONG["orange"], label="RW_Direct")
    ax.set_xlabel("Epsilon")
    ax.set_ylabel("% Rules Homogeneous")
    ax.set_ylim(0, 105)
    ax.set_title(f"Homogeneity Classification Rate — Varying Epsilon ({ds_label})")
    ax.legend()
    ax.grid(True, alpha=0.3)
    save_pdf(fig, f"ablation_{ds_prefix}_homogeneity_impact_epsilon.pdf")

    # ── 6. Param Impact — Delta ───────────────────────────────────────
    fig, ax = plt.subplots(figsize=(7, 4.5))
    if len(delta_fp) > 0:
        ax.plot(delta_fp["delta_pct"], delta_fp["homogeneity_rate"],
                marker="s", linewidth=2.5, color=WONG["blue"], label="FPGrowth")
    if len(delta_rw) > 0:
        ax.plot(delta_rw["delta_pct"], delta_rw["homogeneity_rate"],
                marker="o", linewidth=2.5, color=WONG["orange"], label="RW_Direct")
    ax.set_xlabel("Delta (% of Dataset)")
    ax.set_ylabel("% Rules Homogeneous")
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:.0f}%"))
    ax.set_ylim(0, 105)
    ax.set_title(f"Homogeneity Classification Rate — Varying Delta ({ds_label})")
    ax.legend()
    ax.grid(True, alpha=0.3)
    save_pdf(fig, f"ablation_{ds_prefix}_homogeneity_impact_delta.pdf")

print("✅ Helper functions defined")


## SO Ablation Figures (6 PDFs)


In [ ]:
print("📊 Generating SO ablation figures...")
generate_ablation_figures(SO_ABLATION_CSV, "so", "SO Dataset")
print("Done!\n")


## ACS Ablation Figures (6 PDFs)


In [ ]:
print("📊 Generating ACS ablation figures...")
generate_ablation_figures(ACS_ABLATION_CSV, "acs", "ACS Dataset")
print("Done!\n")


## Benchmark Figures — Problem 2 & 3 (2 PDFs)


In [ ]:
print("📊 Generating benchmark figures...")

import ast as _ast

def _full_rule_label(cond_str, treat_str):
    """Return (condition_text, treatment_text) with full readable strings."""
    try:
        c = _ast.literal_eval(cond_str)
        t = _ast.literal_eval(treat_str)
    except Exception:
        return cond_str, treat_str
    return (", ".join(f"{k} = {v}" for k, v in c.items()),
            ", ".join(f"{k} = {v}" for k, v in t.items()))


def _make_benchmark_figure(df, x_col, x_label, title, pdf_name):
    """Create a benchmark chart with short R1..R10 in the legend
    and a reference table below that shows full Condition → Treatment."""

    # Collect rule info for the table
    rule_info = []  # [(rule_id, cond_text, treat_text, color)]

    fig = plt.figure(figsize=(10, 9))
    # Top: the chart (takes ~60% of height)
    ax = fig.add_axes([0.08, 0.38, 0.62, 0.55])

    for i, (rule_id, grp) in enumerate(df.groupby("Rule_ID")):
        grp = grp.sort_values(x_col)
        color = WONG_LIST[i % len(WONG_LIST)]
        short = f"R{rule_id}"
        ax.plot(grp[x_col], grp["Runtime_Seconds"],
                marker="o", linewidth=2, color=color, label=short)

        cond_text, treat_text = _full_rule_label(
            grp["Condition"].iloc[0], grp["Treatment"].iloc[0])
        rule_info.append((f"R{rule_id}", cond_text, treat_text, color))

    ax.set_yscale("log")
    ax.set_xlabel(x_label)
    ax.set_ylabel("Runtime (seconds, log scale)")
    ax.set_title(title)
    ax.legend(loc="upper left", fontsize=9, frameon=True, ncol=2)
    ax.grid(True, alpha=0.3, which="both")

    # Bottom: reference table
    # Build table text as columns: ID | Condition | Treatment
    col_labels = ["ID", "Condition", "Treatment"]
    cell_text = [[rid, cond, treat] for rid, cond, treat, _ in rule_info]
    cell_colors = [["#f0f0f0" if j == 0 else "white" for j in range(3)]
                   for _ in rule_info]
    # Color the ID cell to match the line
    for idx, (_, _, _, c) in enumerate(rule_info):
        cell_colors[idx][0] = c + "30"  # 30 = ~19% alpha in hex

    tab_ax = fig.add_axes([0.03, 0.01, 0.94, 0.30])
    tab_ax.axis("off")
    table = tab_ax.table(
        cellText=cell_text,
        colLabels=col_labels,
        colWidths=[0.06, 0.42, 0.52],
        loc="upper center",
        cellLoc="left",
    )
    table.auto_set_font_size(False)
    table.set_fontsize(8)
    table.scale(1, 1.3)

    # Style header row
    for j in range(3):
        cell = table[0, j]
        cell.set_text_props(weight="bold")
        cell.set_facecolor("#d0d0d0")

    # Color ID cells to match line colors
    for idx, (_, _, _, c) in enumerate(rule_info):
        table[idx + 1, 0].set_facecolor(c + "30")

    save_pdf(fig, pdf_name)


# ── Problem 2: Runtime vs Epsilon (10 rules) ─────────────────────────
p2 = pd.read_csv(BENCH_P2_CSV)
_make_benchmark_figure(
    p2, x_col="Epsilon",
    x_label="ε (fixed for each experiment)",
    title="Runtime vs ε — Finding Largest Subgroup Size (Problem 2)",
    pdf_name="benchmark_problem2_runtime_vs_epsilon.pdf")

# ── Problem 3: Runtime vs Delta (10 rules) ───────────────────────────
p3 = pd.read_csv(BENCH_P3_CSV)
_make_benchmark_figure(
    p3, x_col="Delta",
    x_label="δ (minimum subgroup size)",
    title="Runtime vs δ — Finding Smallest Epsilon Threshold (Problem 3)",
    pdf_name="benchmark_problem3_runtime_vs_delta.pdf")

print("Done!\n")


## Summary


In [ ]:
# List all generated PDFs
pdfs = sorted([f for f in os.listdir(OUTPUT_DIR) if f.endswith(".pdf")])
print(f"🎉 Total PDF figures generated: {len(pdfs)}\n")
for f in pdfs:
    print(f"  📄 {f}")
